In [1]:
import json
import requests
import os
from datetime import datetime
from dotenv import load_dotenv

In [2]:
load_dotenv("../.env")

CHAT_URL = os.getenv("CHAT_URL")
API_KEY = os.getenv("API_KEY")
MODEL = os.getenv("MODEL")

In [3]:
def calculator(expression:str) -> str:
    allowed = set("1234567890+-*/.() ")
    if not set(expression) <= allowed:
        return "Error: disallowed characters in the expression."
    try:
        result = eval(expression, {"__builtins__": {}}, {})
        return str(round(result, 2))
    except Exception as e:
        return f"Error: {e}"

In [4]:
def get_current_time() -> str:
    return datetime.now().strftime("%A, %Y-%m-%d %H:%M:%S")

In [5]:
TOOLS_SPEC = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Evaluate a basic arithmatic expression",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {"expression": {"type": "string"}},
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current local date and time.",
            "strict": True,
            "parameters": {"type": "object", "properties": {}},
        },
    },
]

AVAILABLE_FUNCTiONS = {"calculator": calculator, "get_current_time": get_current_time}

In [6]:
def call_model(messages, tools=None):
    payload = {"model": MODEL, "messages": messages, "stream": False}
    if tools:
        payload["tools"] = tools
    headers = {"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"}
    response = requests.post(CHAT_URL, headers=headers, json=payload, timeout=60)
    print("model response status code", response.status_code)
    return response.json()

In [7]:
def run_agent(user_question: str, max_steps: int = 5) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "You are a careful assistant. "
                "Use tools whenever they help (math, current time etc.). "
                "When you have everything you need, give a final, direct "
                "answer without calling any more tools. "
                "Never include arguments, descriptions, or reason fields "
                "for functions that do not define parameters in their "
                "specification."
            ), 
        },
        {"role": "user", "content": user_question},
    ]
    for step in range(max_steps):
        print(f"\n[step {step + 1}]")

        data = call_model(messages=messages, tools=TOOLS_SPEC)
        message = data["choices"][0]["message"]
        messages.append(message)

        tool_calls = message.get("tool_calls")
        if not tool_calls:
            return message["content"]

        print(f"\nModel requested tool calls: {json.dumps(tool_calls, indent=2)}")

        for tool_call in tool_calls:
            fn_name = tool_call["function"]["name"]
            fn_args = json.loads(tool_call["function"]["arguments"] or "{}")

            fn_spec = next(t for t in TOOLS_SPEC if t["function"]["name"] == fn_name)
            allowed_props = fn_spec["function"].get("parameters", {}).get("properties", {})
            clean_args = {k: v for k, v in fn_args.items() if k in allowed_props}

            print(f"    Calling {fn_name}({clean_args})")
            result = AVAILABLE_FUNCTiONS[fn_name](**clean_args)
            print(f"    Result: {result}")

            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call["id"],
                    "content": str(result),
                }
            )
    return "Reached max steps without a final answer."

In [8]:
question = "What is a 15 percent tip on $84.50 calculated at the current time?"
print(f"Question: {question}")

answer = run_agent(question)
print(f"\nAnswer: {answer}")

Question: What is a 15 percent tip on $84.50 calculated at the current time?

[step 1]
model response status code 200

Model requested tool calls: [
  {
    "id": "call_3156277",
    "type": "function",
    "function": {
      "name": "get_current_time",
      "arguments": "{\"reason\":\"Determining the current date and time as requested by the user's prompt.\"}"
    }
  }
]
    Calling get_current_time({})
    Result: Sunday, 2026-09-06 17:29:09

[step 2]
model response status code 200

Model requested tool calls: [
  {
    "id": "call_2958562",
    "type": "function",
    "function": {
      "name": "calculator",
      "arguments": "{\"expression\":\"84.50 * 0.15\"}"
    }
  }
]
    Calling calculator({'expression': '84.50 * 0.15'})
    Result: 12.67

[step 3]
model response status code 200

Answer: A 15 percent tip on $84.50 is $12.68 (calculated as $12.675, rounded to the nearest cent).
